# 2110446 DATA SCIENCE AND DATA ENGINEERING

## **Unit 06:** Data Extraction

- **Problem:** Web Scraping (`07_scrape_01_2025s2`)
- **Author:** Worralop Srichainont
- **Year:** 2025 (Semester 2)

# Dependencies

In [40]:
import gdown

from bs4 import BeautifulSoup
from IPython.display import HTML, display

# Load Files

In [41]:
HTML_FILE_URL = "https://drive.google.com/uc?id=1f9HaOvz-2eg5WIES5m4A8jF7N0h4yGVL"
HTML_FILE_PATH = "2566.html"

In [42]:
gdown.download(HTML_FILE_URL, HTML_FILE_PATH, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=1f9HaOvz-2eg5WIES5m4A8jF7N0h4yGVL
To: /content/2566.html
100%|██████████| 329k/329k [00:00<00:00, 717MB/s]


'2566.html'

# Utility Function

## Filter Buddhist Holy Day

As we can see in the HTML file, all Buddhist holy day (วันพระ) are indicated in `bud-day` class and the information inside is indicated in `bud-day-column`. For example.

```
<div class="bud-day">
    <div class="bud-day-col">วันศุกร์ที่ 6 มกราคม 2566</div>
    <div class="bud-day-col">ขึ้น ๑๕ ค่ำ เดือนยี่(๒) ปีขาล</div>
    <div class="bud-day-col"></div>
</div>
```

In [43]:
BUDDHIST_DAY_COLUMN_NAME = "div.bud-day-col"

- Make sure to open the file using `UTF-8` encoder to support Thai language.
- Use `lxml` engine to parse HTML file.
- Use `select()` method to filter only section with `<div class="bud-day-column">` tag

In [44]:
def get_buddhist_day_columns(file_path=HTML_FILE_PATH):
    with open(file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "lxml")
    return soup.select(BUDDHIST_DAY_COLUMN_NAME)

Visualize the result from the utility function

In [45]:
N_DISPLAY_ROWS = 15
result = get_buddhist_day_columns()

In [46]:
for item in result[:N_DISPLAY_ROWS]:
    display(HTML(str(item)))

## Get Day Index

Create a function that checks the day inside a string and return an index
- Return `0` if the text starts with `วันจันทร์`
- Return `1` if the text starts with `วันอังคาร`
- Return `2` if the text starts with `วันพุธ`
- Return `3` if the text starts with `วันพฤหัสบดี`
- Return `4` if the text starts with `วันศุกร์`
- Return `5` if the text starts with `วันเสาร์`
- Return `6` if the text starts with `วันอาทิตย์`
- Otherwise, return `-1`

In [47]:
DAY_PREFIX = "วัน"
DAY_NAMES = (
    "วันจันทร์",
    "วันอังคาร",
    "วันพุธ",
    "วันพฤหัสบดี",
    "วันศุกร์",
    "วันเสาร์",
    "วันอาทิตย์",
)

In [48]:
def get_day_idx(text):
    if not text.startswith(DAY_PREFIX):
        return -1

    for idx, day_name in enumerate(DAY_NAMES):
        if text.startswith(day_name):
            return idx
    return -1

# Problem `Q1`

Write function `Q1` that counts the number of Buddhist holy days (วันพระ) in each year provided.

The function will count the Buddhist holy days in each specified year and return a list that contains the number of Buddhist holy days for each year, from the first Monday (index 0) until the following Sunday.

Initialize a list to count amount of Buddhist holy days in Monday to Sunday.

In [49]:
buddhist_day_count = [0, 0, 0, 0, 0, 0, 0]

Implement the counting logic

In [50]:
for item in get_buddhist_day_columns():
    # Skip if current item has no text
    if not isinstance(item.text, str):
        continue

    # Skip if the current item is not a date
    idx = get_day_idx(item.text.strip())
    if idx == -1:
        continue

    # Update the counting list
    buddhist_day_count[idx] += 1

Visualize the result

In [51]:
print(buddhist_day_count)
for idx, day_name in enumerate(DAY_NAMES):
    print(f"- There are {buddhist_day_count[idx]} Buddhist holy day in {day_name}")

[8, 6, 8, 6, 6, 7, 9]
- There are 8 Buddhist holy day in วันจันทร์
- There are 6 Buddhist holy day in วันอังคาร
- There are 8 Buddhist holy day in วันพุธ
- There are 6 Buddhist holy day in วันพฤหัสบดี
- There are 6 Buddhist holy day in วันศุกร์
- There are 7 Buddhist holy day in วันเสาร์
- There are 9 Buddhist holy day in วันอาทิตย์


# Problem `Q2`

Write function `Q2` that finds Visakha Bucha Day (วันวิสาขบูชา) for each year provided in the specified HTML file.

The function will return a string representing the Visakha Bucha Day in the format `(dayWeek, monthNumber, month, year)`, for example, `วันเสาร์ที่ 3 มิถุนายน 2566`.

Here is the HTML section where are we finding for.

```
<div class="bud-day">
    <div class="bud-day-col">วันเสาร์ที่ 3 มิถุนายน 2566</div>
    <div class="bud-day-col">
        ขึ้น ๑๕ ค่ำ เดือนเจ็ด(๗) ปีเถาะ
    </div>
    <div class="bud-day-col">
        (วันเฉลิมฯ พระบรมราชินี,
        <a
        href="https://www.myhora.com/%E0%B8%9B%E0%B8%8F%E0%B8%B4%E0%B8%97%E0%B8%B4%E0%B8%99/%e0%b8%a7%e0%b8%b1%e0%b8%99%e0%b8%a7%e0%b8%b4%e0%b8%aa%e0%b8%b2%e0%b8%82%e0%b8%9a%e0%b8%b9%e0%b8%8a%e0%b8%b2.aspx"
        target="blank"
        title="วันวิสาขบูชา"
        >วันวิสาขบูชา</a
        >)
    </div>
</div>
```

First, define the target day you are going to search

In [52]:
TARGET_DAY = "วันวิสาขบูชา"

According to the HTML file, each day are grouped in the `bud-day` tag. Each `bud-day` tag always has 3 `bud-day-col` tags which contains

- The first `bud-day-col` tag is date (e.g. วันเสาร์ที่ 3 มิถุนายน 2566).
- The second `bud-day-col` tag is the lunar phase (e.g. ขึ้น ๑๕ ค่ำ เดือนเจ็ด(๗) ปีเถาะ).
- The third `bud-day-col` tag is a hyperlink to the holiday info (e.g. วันวิสาขบูชา).
    - If the current date is not holiday, this field will be empty.

Therefore, it can be implemented in Python code as follows.

In [53]:
# Initialize the variable to store the holiday date
holiday_date = ""

# Get text for all buddhist day column tag field
buddhist_day_texts = [item.text for item in get_buddhist_day_columns()]

In [54]:
for idx in range(0, len(buddhist_day_texts), 3):
    # Get date and holiday info field
    date = buddhist_day_texts[idx]
    holiday_info = buddhist_day_texts[idx + 2]

    # If the holiday info are empty, skip it
    if not holiday_info:
        continue

    # If the current holiday is not the target, skip it
    if TARGET_DAY not in holiday_info.strip():
        continue

    # Update the holiday date variable field
    holiday_date = date
    break

Visualize the result.

In [55]:
print(f"{TARGET_DAY} is in {holiday_date}")

วันวิสาขบูชา is in วันเสาร์ที่ 3 มิถุนายน 2566


# Grader

## Solution

Write all Python grader code solution here.

In [56]:
# Constants
BUDDHIST_DAY_COLUMN_NAME = "div.bud-day-col"
DAY_PREFIX = "วัน"
DAY_NAMES = (
    "วันจันทร์",
    "วันอังคาร",
    "วันพุธ",
    "วันพฤหัสบดี",
    "วันศุกร์",
    "วันเสาร์",
    "วันอาทิตย์",
)
TARGET_DAY = "วันวิสาขบูชา"


# Utility Functions
def get_buddhist_day_columns(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "lxml")
    return soup.select(BUDDHIST_DAY_COLUMN_NAME)


def get_day_idx(text):
    if not text.startswith(DAY_PREFIX):
        return -1

    for idx, day_name in enumerate(DAY_NAMES):
        if text.startswith(day_name):
            return idx
    return -1


# Assignment Tasks
def Q1(file_path):
    buddhist_day_count = [0, 0, 0, 0, 0, 0, 0]

    for item in get_buddhist_day_columns(file_path):
        # Skip if current item has no text
        if not isinstance(item.text, str):
            continue

        # Skip if the current item is not a date
        idx = get_day_idx(item.text.strip())
        if idx == -1:
            continue

        # Update the counting list
        buddhist_day_count[idx] += 1

    return buddhist_day_count


def Q2(file_path, target_day=TARGET_DAY):
    # Initialize the variable to store the holiday date
    holiday_date = ""

    # Get text for all buddhist day column tag field
    buddhist_day_texts = [item.text for item in get_buddhist_day_columns(file_path)]

    for idx in range(0, len(buddhist_day_texts), 3):
        # Get date and holiday info field
        date = buddhist_day_texts[idx]
        holiday_info = buddhist_day_texts[idx + 2]

        # If the holiday info are empty, skip it
        if not holiday_info:
            continue

        # If the current holiday is not the target, skip it
        if target_day not in holiday_info.strip():
            continue

        # Update the holiday date variable field
        holiday_date = date
        break

    return holiday_date

## Run Test

Create a utility class `WebScrapingTest` to check if each method in the class works correctly.

Use `assert`, `try` and `except` to check if the result matches.

In [57]:
class WebScrapingTest:
    @staticmethod
    def run_tests():
        TESTCASES = (
            {"func_name": "Q1", "key": [8, 6, 8, 6, 6, 7, 9], "args": ("2566.html",)},
            {
                "func_name": "Q2",
                "key": "วันเสาร์ที่ 3 มิถุนายน 2566",
                "args": ("2566.html",),
            },
        )

        for testcase in TESTCASES:
            func_name = testcase["func_name"]
            key = testcase["key"]
            args = testcase["args"]
            WebScrapingTest.run_test(func_name, key, *args)

    @staticmethod
    def run_test(func_name, key, *args, **kwargs):
        try:
            result = globals().get(func_name)(*args, **kwargs)
            assert result == key, f"Expected {key} but got {result}"
        except Exception as e:
            print(f"{func_name} failed with {e}")
            return
        print(f"{func_name} passed.")

In [58]:
WebScrapingTest.run_tests()

Q1 passed.
Q2 passed.
